# 01 — Building the IPO panel

**Question.** Are companies that already had a Wikipedia article before going
public *more underpriced* on their first trading day?

A hobby project, built with **only free, unauthenticated data sources**. No
WRDS, no CRSP, no Compustat, no SDC Platinum, no API keys.

### Design choices, and why

| Choice | Why |
|---|---|
| Sample window 2014–2024 | Free price sources have purged firms that delisted out of older windows |
| Yahoo Finance chart API for first-day close | Only free source with usable IPO-date closes |
| Jay Ritter's `IPO-age.xlsx` for the universe | Free, and carries offer date, VC flag and founding year |
| Name matching + redirect handling, with a human-review queue | Parent/subsidiary/product relationships are judgment calls, so they get escalated rather than guessed |
| Controls: `vc`, `tech`, `log(proceeds)`, year FE | Underwriter reputation needs prospectus name-matching; no free news database spans the window |

Because the control set is narrow, the headline number is best read as an
unconditional difference between the two groups rather than a conditional
effect. Notebook 03 makes that explicit.

In [1]:
import json, sys, subprocess
from pathlib import Path
import pipeline as P

print("data directory:", P.ROOT)
for f in sorted((P.RAW).iterdir()):
    print(f"  {f.name:26s} {f.stat().st_size/1e6:8.2f} MB")

data directory: /Users/daniel/Documents/coding_stuff/IPOTracker/research/data/underpricing
  IPO-age.xlsx                   1.35 MB
  Underwriter-Rank.xls           0.30 MB
  cik-lookup-data.txt           40.08 MB
  company_tickers.json           0.80 MB


## Source 1 — the IPO universe

Jay Ritter's `IPO-age.xlsx` (University of Florida) lists 16,031 US IPOs from
1975 to 2025 with offer date, ticker, CUSIP, an ADR flag, a **VC-backing
flag**, post-issue shares and founding year — a free substitute for the
commercial IPO databases that normally supply this.

Screens applied, following standard IPO-sample practice: drop ADRs, drop
blank-check/SPAC vehicles (ticker suffix `U`, or "Acquisition"/"Blank Check"
in the name), require a ticker.

In [2]:
univ = P.load_universe(2014, 2024)
print(f"universe after screens: {len(univ)} IPOs")

import collections
by_year = collections.Counter(f["year"] for f in univ)
for y in sorted(by_year):
    print(f"  {y}  {by_year[y]:4d}  {'#' * (by_year[y] // 6)}")

universe after screens: 1998 IPOs
  2014   269  ############################################
  2015   153  #########################
  2016    88  ##############
  2017   149  ########################
  2018   165  ###########################
  2019   144  ########################
  2020   225  #####################################
  2021   451  ###########################################################################
  2022    93  ###############
  2023   107  #################
  2024   154  #########################


## Source 2 — CIK resolution

`company_tickers.json` covers only ~10,400 *current* filers, which misses every
firm that has since delisted or been renamed. `cik-lookup-data.txt` (40 MB)
carries all historical company names, and lifts the match rate from 47% to 95%.

In [3]:
hits = 0
for f in univ:
    f["cik"], f["cik_via"] = P.resolve_cik(f)
    hits += bool(f["cik"])
print(f"CIK resolved: {hits}/{len(univ)} ({hits/len(univ):.1%})")
print("  by method:", collections.Counter(f["cik_via"] for f in univ))

CIK resolved: 1899/1998 (95.0%)
  by method: Counter({'ticker': 1072, 'name': 794, 'miss': 99, 'name2': 33})


## Sources 3 & 4 — offer price and first-day close

**Offer price** comes from the 424B prospectus cover on EDGAR, located by
searching a firm's filing history for a `424B*` within 25 days of the offer
date, then regexing the cover page. Only the first 900 KB of each document is
fetched, which keeps the crawl to a few hundred MB instead of several GB.

**First-day close** comes from Yahoo's chart API. Two traps handled:

1. Yahoo's `close` series is **split-adjusted**, so a later reverse split would
   inflate a historical price. We un-adjust using the split events the API
   returns.
2. Yahoo returns placeholder records (`instrumentType: MUTUALFUND`,
   `firstTradeDate: null`) for delisted tickers rather than an error. Those are
   rejected, and we require the matched bar to be within 7 days of the offer
   date so a series that starts years late cannot masquerade as day one.

Both stages are cached per firm on disk, so the crawl is resumable.
Run `uv run --with openpyxl python build_panel.py` to (re)build.

In [4]:
panel_path = P.OUT / "panel.json"
if not panel_path.exists():
    raise SystemExit("run build_panel.py first")
rows = json.loads(panel_path.read_text())
print(f"panel rows: {len(rows)}")
print(f"attrition: {len(univ)} universe -> {len(rows)} with price AND offer "
      f"({len(rows)/len(univ):.0%})")

panel rows: 899
attrition: 1998 universe -> 899 with price AND offer (45%)


### Why the attrition is what it is

The dominant loss is **survivorship in free price data**. Firms that delisted —
acquired, taken private, or bankrupt — have been purged from Yahoo. Spot checks
on 2014 failures (GlycoMimetics, CHC Group, EP Energy, Cypress Energy Partners)
confirm these are genuine delistings, not lookup bugs.

This biases the sample toward survivors. It is stated as a limitation in
notebook 03 rather than corrected, because correcting it requires exactly the
paid data this project is avoiding.

In [5]:
by_year_panel = collections.Counter(r["year"] for r in rows)
print("coverage by year (kept / universe):")
for y in sorted(by_year):
    k = by_year_panel.get(y, 0)
    print(f"  {y}  {k:4d}/{by_year[y]:4d}  {k/by_year[y]:5.0%}")

coverage by year (kept / universe):
  2014    45/ 269    17%
  2015    38/ 153    25%
  2016    34/  88    39%
  2017    50/ 149    34%
  2018    74/ 165    45%
  2019    63/ 144    44%
  2020   107/ 225    48%
  2021   232/ 451    51%
  2022    59/  93    63%
  2023    71/ 107    66%
  2024   126/ 154    82%
